# C9-dimensionality-reduction — Practice p14 — Solution

In [1]:
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

kv = gensim.downloader.load("glove-wiki-gigaword-100")

WORDS = ["doctor", "lawyer", "teacher", "farmer", "chef", "pilot",
         "castle", "cottage", "tower", "barn", "palace", "hut",
         "coffee", "tea", "juice", "cider", "lemonade", "cocoa",
         "beetle", "moth", "wasp", "ant", "bee", "spider",
         "ruby", "emerald", "sapphire", "pearl", "topaz", "jade"]
V = np.asarray(kv[WORDS], dtype=np.float64)
W = V / np.sqrt((V * V).sum(axis=1, keepdims=True))
assert np.allclose(np.sqrt((W * W).sum(axis=1)), 1.0, atol=1e-12, rtol=0)
assert np.isclose((W * W).sum(), 30.0, atol=1e-9, rtol=0)
U, s, Vt = np.linalg.svd(W, full_matrices=False)

fro2 = float((s**2).sum())
rel_err2 = (fro2 - np.cumsum(s**2)) / fro2
r15 = int(np.argmax(rel_err2 <= 0.15) + 1)
r05 = int(np.argmax(rel_err2 <= 0.05) + 1)
for rank, budget in ((r15, 0.15), (r05, 0.05)):
    assert rel_err2[rank - 1] <= budget
    assert rank == 1 or rel_err2[rank - 2] > budget

S = W @ W.T
W_r = U[:, :r05] @ np.diag(s[:r05]) @ Vt[:r05]
S_r = W_r @ W_r.T
max_gap = float(np.max(np.abs(S - S_r)))
top1 = np.empty(30, dtype=int)
top1_r = np.empty(30, dtype=int)
for i in range(30):
    order = np.argsort(S[i])[::-1]
    top1[i] = order[order != i][0]
    order_r = np.argsort(S_r[i])[::-1]
    top1_r[i] = order_r[order_r != i][0]
changed_mask = top1 != top1_r
n_changed = int(changed_mask.sum())
changed_words = np.asarray(WORDS, dtype=object)[changed_mask].tolist()

W_15 = U[:, :r15] @ np.diag(s[:r15]) @ Vt[:r15]
S_15 = W_15 @ W_15.T
top1_15 = np.empty(30, dtype=int)
for i in range(30):
    order_15 = np.argsort(S_15[i])[::-1]
    top1_15[i] = order_15[order_15 != i][0]
n_changed15 = int((top1 != top1_15).sum())
max_gap15 = float(np.max(np.abs(S - S_15)))
worst_position = np.unravel_index(np.argmax(np.abs(S - S_r)), S.shape)
worst_entry = (int(worst_position[0]), int(worst_position[1]))

print("budget ranks:", r15, r05)
print("0.05 value gap / changed words:", max_gap, n_changed, changed_words)
print("0.15 value gap / changed count:", max_gap15, n_changed15)
print("worst 0.05 entry:", worst_entry, WORDS[worst_entry[0]], WORDS[worst_entry[1]])

budget ranks: 15 22
0.05 value gap / changed words: 0.10995964532049685 1 ['spider']
0.15 value gap / changed count: 0.20230633744227056 4
worst 0.05 entry: (25, 25) emerald emerald


At the $0.05$ budget, the largest entrywise similarity damage is $0.1099596$ but only one top-$1$ neighbor changes (`spider`); the worst value loss is a diagonal self-similarity entry (`emerald`, `emerald`), not a competing neighbor pair.  The $0.15$ budget is not defensible for this nearest-neighbor use without further acceptance criteria: its larger $0.2023063$ value gap changes four of thirty top-$1$ choices, whereas the stricter rank-$22$ version changes one.

### Answer check

In [2]:
assert W.shape == (30, 100) and W.dtype == np.float64
assert np.isclose((W * W).sum(), 30.0, atol=1e-9, rtol=0)
assert r15 == 15 and r05 == 22
assert np.isclose(max_gap, 0.10995964532049685, atol=1e-12, rtol=0)
assert n_changed == 1 and changed_words == ["spider"]
assert worst_entry == (25, 25)
assert np.isclose(max_gap15, 0.20230633744227056, atol=1e-12, rtol=0)
assert n_changed15 == 4